In [56]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from datetime import datetime

In [57]:
file_path = 'UST Data Sample (July - Sept 2024).xlsx'
df = pd.read_excel(file_path)
df

,let me tak,Id,Client,RFQStatus,InstrumentSubGroup,Market Traded Vol (USD),Our Traded Vol,TiedWonVol,Num of Dealers,Trader,...,SettlementDays,SettlementDate,SettlementStops,RFQ CreateTime,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket
0,2024-09-30,66FA7D45432800D20002,Client1,Done,OFFTHERUN,100000000,100000000,100000000,6,Trader1,...,1,2024-10-01,T+1,2024-09-30 06:28:00,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y
1,2024-09-30,TRSY_20240930_15961,Client2,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,5,Trader2,...,1,2024-10-01,T+1,2024-09-30 16:00:00,2024-09-30 16:00:00,1,0.95,1-5M,7-10 YR,7y-10y
2,2024-09-30,TRSY_20240930_314,Client2,Done,ONTHERUN,53000000,53000000,53000000,5,Trader3,...,1,2024-10-01,T+1,2024-09-30 15:46:00,2024-09-30 15:46:00,1,0.07,50+M,7-10 YR,7y-10y
3,2024-09-30,66FAEA57432800020001,Client3,Done,OFFTHERUN,500000,500000,500000,5,Trader4,...,1,2024-10-01,T+1,2024-09-30 14:14:00,2024-09-30 14:14:00,1,0.03,500K-1M,0-1 YR,<18mos
4,2024-09-30,66FAC35D455C00210006,Client4,Done,OFFTHERUN,1058000,1058000,1058000,5,Trader2,...,1,2024-10-01,T+1,2024-09-30 11:27:00,2024-09-30 11:27:00,1,0.03,1-5M,5-7 YR,3y-5y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66676,2024-07-01,E68302E5454C00010001,Client33,CustomerTimeOut,ONTHERUN,0,0,0,1,Trader4,...,1,2024-07-02,T+1,2024-07-01 15:26:00,2024-07-01 15:27:00,1,0.04,1-5M,BILLS,<18mos
66677,2024-07-01,E68305CA57D000080001,Client33,CustomerTimeOut,ONTHERUN,0,0,0,1,Trader4,...,1,2024-07-02,T+1,2024-07-01 15:38:00,2024-07-01 15:40:00,1,0.04,1-5M,BILLS,<18mos
66678,2024-07-01,E683099657D000010001,Client33,CustomerTimeOut,ONTHERUN,0,0,0,1,Trader4,...,1,2024-07-02,T+1,2024-07-01 15:55:00,2024-07-01 15:56:00,1,0.04,1-5M,BILLS,<18mos
66679,2024-07-01,6682BE1857D0000C0002,Client271,CustomerTimeOut,OFFTHERUN,0,0,0,15,Trader4,...,1,2024-07-02,T+1,2024-07-01 10:33:00,2024-07-01 10:34:00,0,4.99,10-25M,0-1 YR,<18mos


In [58]:
# Modify the year < 2000
df['Maturity Date'] = df['Maturity Date'].apply(lambda x: x if x.year >= 2000 else x.replace(year=x.year + 100))
df['Maturity Date'].unique()

<DatetimeArray>
['2030-04-30 00:00:00', '2034-02-15 00:00:00', '2031-09-30 00:00:00',
 '2025-09-15 00:00:00', '2028-06-30 00:00:00', '2024-10-29 00:00:00',
 '2029-09-30 00:00:00', '2027-09-15 00:00:00', '2034-08-15 00:00:00',
 '2044-08-15 00:00:00',
 ...
 '2034-11-15 00:00:00', '2024-07-23 00:00:00', '2024-07-18 00:00:00',
 '2037-08-15 00:00:00', '2024-07-16 00:00:00', '2024-07-15 00:00:00',
 '2024-07-11 00:00:00', '2024-07-09 00:00:00', '2024-07-05 00:00:00',
 '2024-07-02 00:00:00']
Length: 319, dtype: datetime64[ns]

In [59]:
missing_values = df.isnull().sum()
missing_values

let me tak                     0
Id                             0
Client                         0
RFQStatus                      0
InstrumentSubGroup             0
Market Traded Vol (USD)        0
Our Traded Vol                 0
TiedWonVol                     0
Num of Dealers                 0
Trader                         0
Sales                          0
Covered Vol                    0
ActionStr                      0
Tier                           0
InstrumentCode                 0
InstrumentDescription          0
AssetClass                     0
Market                         0
LegNo                          0
Maturity Date                  0
RFQNumberOfQuotes              0
Buy/Sell                       0
Best Bid Price                 0
Mid Price                      0
Best Ask Price                 0
Deal Value                     0
AwayFromMid                    0
TickIdentifier               445
Deal Spread                    0
Cover                          0
CoverPnL  

In [60]:
df.dropna(inplace=True)

In [61]:
df = df[df['AssetClass'] != 'TFRNS']

In [62]:
# Function to calculate years to maturity
def calculate_years_to_maturity(row):
    maturity_date = pd.to_datetime(row['Maturity Date'])
    trade_date = pd.to_datetime(row['let me tak'])
    return (maturity_date - trade_date).days / 365.25

df['Years to Maturity'] = df.apply(calculate_years_to_maturity, axis=1)


def extract_yield(description):
    match = re.search(r"(\d+\.\d+)", description)
    return float(match.group(1)) if match else np.nan  # Return NaN if no match is found

df['Yield'] = df['InstrumentDescription'].apply(extract_yield)

def yield_to_price(ytm, years_to_maturity, face_value=100):
    if pd.isna(ytm) or pd.isna(years_to_maturity):
        return np.nan
    return face_value / ((1 + 0.01 * ytm) ** years_to_maturity)

def bill_price(face_value, deal_price, days_to_maturity):
    if pd.isna(deal_price) or pd.isna(days_to_maturity):
        return np.nan
    discount_yield = ((face_value - deal_price) / face_value) * (360 / days_to_maturity)
    return face_value * (1 - (discount_yield / 100) * (days_to_maturity / 360))

# Columns to convert
columns_to_convert = ['Best Bid Price', 'Mid Price', 'Best Ask Price', 'Deal Value', 'Cover']

for column in columns_to_convert:
    df[f'Converted {column}'] = df.apply(
        lambda row: (
            # yield_to_price(row['Yield'], row['Years to Maturity']) if row['AssetClass'] == 'STRIPS'
            # else (
                bill_price(100, row[column], (pd.to_datetime(row['Maturity Date']) - pd.to_datetime(row['let me tak'])).days)
                if (row['AssetClass'] == 'BILLS' or row['AssetClass'] == 'STRIPS')else row[column]
            # )
        ),
        axis=1
    )

print(df[['InstrumentDescription', 'AssetClass', 'Yield', 'Years to Maturity'] + [f'Converted {col}' for col in columns_to_convert]])

/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/1194386698.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Years to Maturity'] = df.apply(calculate_years_to_maturity, axis=1)
/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/1194386698.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Yield'] = df['InstrumentDescription'].apply(extract_yield)
/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/1194386698.py:31: SettingWithCopyWarning: 
A value is t

        InstrumentDescription AssetClass  Yield  Years to Maturity  \
0           T  3.500 04/30/30      BONDS  3.500           5.579740   
1           T  4.000 02/15/34      BONDS  4.000           9.377139   
2      T  3.625 09/30/31  7YR      BONDS  3.625           6.997947   
3           T  3.500 09/15/25      BONDS  3.500           0.958248   
4           T  4.000 06/30/28      BONDS  4.000           3.748118   
...                       ...        ...    ...                ...   
66676        B  09/26/24  3MO      BILLS    NaN           0.238193   
66677        B  09/26/24  3MO      BILLS    NaN           0.238193   
66678        B  09/26/24  3MO      BILLS    NaN           0.238193   
66679       T  1.000 12/15/24      BONDS  1.000           0.457221   
66680             B  07/02/24      BILLS    NaN           0.002738   

       Converted Best Bid Price  Converted Mid Price  \
0                     99.539062            99.550781   
1                    101.712388           101.7

/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/1194386698.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'Converted {column}'] = df.apply(


In [63]:
# Filter the DataFrame for AssetClass that is either BILLS or STRIPS
bills_strips_df = df[df['AssetClass'].isin(['BILLS', 'STRIPS', 'BONDS'])]

# Select only the columns that are related to converted prices
columns_to_display = ['InstrumentDescription', 'AssetClass', 'Yield', 'Years to Maturity'] + [f'Converted {col}' for col in columns_to_convert]

# Display the filtered DataFrame with the desired columns
print(bills_strips_df[columns_to_display].head(30))

      InstrumentDescription AssetClass  Yield  Years to Maturity  \
0         T  3.500 04/30/30      BONDS  3.500           5.579740   
1         T  4.000 02/15/34      BONDS  4.000           9.377139   
2    T  3.625 09/30/31  7YR      BONDS  3.625           6.997947   
3         T  3.500 09/15/25      BONDS  3.500           0.958248   
4         T  4.000 06/30/28      BONDS  4.000           3.748118   
5          B  10/29/24  1MO      BILLS    NaN           0.079398   
6    T  3.500 09/30/29  5YR      BONDS  3.500           4.999316   
7    T  3.625 09/30/31  7YR      BONDS  3.625           6.997947   
8    T  3.375 09/15/27  3YR      BONDS  3.375           2.956879   
9   T  3.875 08/15/34  10YR      BONDS  3.875           9.872690   
10   T  3.500 09/30/29  5YR      BONDS  3.500           4.999316   
11  T  4.125 08/15/44  20YR      BONDS  4.125          19.874059   
12  T  3.875 08/15/34  10YR      BONDS  3.875           9.872690   
13  T  4.250 08/15/54  30YR      BONDS  4.250   

In [64]:
df['Spread'] = df['Converted Best Ask Price'] - df['Converted Best Bid Price']

/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/573616645.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Spread'] = df['Converted Best Ask Price'] - df['Converted Best Bid Price']


## slippage

In [65]:
def calculate_slippage_price(row):
    if row['Buy/Sell'] == 'Buy':
        return (row['Converted Deal Value'] - row['Converted Mid Price'])
    elif row['Buy/Sell'] == 'Sell':
        return (row['Converted Mid Price'] - row['Converted Deal Value'])
    return None

df['Slippage'] = df.apply(calculate_slippage_price, axis=1)

/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/1307256940.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Slippage'] = df.apply(calculate_slippage_price, axis=1)


# VPIN

## Volume Bucketing

In [85]:
import copy
df['Sign'] = df['Buy/Sell'].replace({'Buy': 1, 'Sell': -1})
df

/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/3682741170.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Sign'] = df['Buy/Sell'].replace({'Buy': 1, 'Sell': -1})
/var/folders/16/fkny30t96j9cdtccgcqbt79h0000gn/T/ipykernel_63523/3682741170.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sign'] = df['Buy/Sell'].replace({'Buy': 1, 'Sell': -1})


,let me tak,Id,Client,RFQStatus,InstrumentSubGroup,Market Traded Vol (USD),Our Traded Vol,TiedWonVol,Num of Dealers,Trader,...,Years to Maturity,Yield,Converted Best Bid Price,Converted Mid Price,Converted Best Ask Price,Converted Deal Value,Converted Cover,Spread,Slippage,Sign
0,2024-09-30,66FA7D45432800D20002,Client1,Done,OFFTHERUN,100000000,100000000,100000000,6,Trader1,...,5.579740,3.500,99.539062,99.550781,99.562500,99.558594,99.558594,0.023438,-0.007812,-1
1,2024-09-30,TRSY_20240930_15961,Client2,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,5,Trader2,...,9.377139,4.000,101.712388,101.724107,101.735826,101.726562,101.726562,0.023438,-0.002456,-1
2,2024-09-30,TRSY_20240930_314,Client2,Done,ONTHERUN,53000000,53000000,53000000,5,Trader3,...,6.997947,3.625,99.761719,99.765625,99.769531,99.761719,99.761719,0.007812,0.003906,-1
3,2024-09-30,66FAEA57432800020001,Client3,Done,OFFTHERUN,500000,500000,500000,5,Trader4,...,0.958248,3.500,99.535156,99.539062,99.542969,99.554688,99.554688,0.007812,-0.015625,-1
4,2024-09-30,66FAC35D455C00210006,Client4,Done,OFFTHERUN,1058000,1058000,1058000,5,Trader2,...,3.748118,4.000,101.601562,101.609375,101.617188,101.605469,101.605469,0.015625,-0.003906,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66676,2024-07-01,E68302E5454C00010001,Client33,CustomerTimeOut,ONTHERUN,0,0,0,1,Trader4,...,0.238193,NaN,99.052250,99.052225,99.052200,99.052175,99.000000,-0.000050,0.000050,-1
66677,2024-07-01,E68305CA57D000080001,Client33,CustomerTimeOut,ONTHERUN,0,0,0,1,Trader4,...,0.238193,NaN,99.052250,99.052225,99.052200,99.052175,99.000000,-0.000050,0.000050,-1
66678,2024-07-01,E683099657D000010001,Client33,CustomerTimeOut,ONTHERUN,0,0,0,1,Trader4,...,0.238193,NaN,99.052250,99.052225,99.052200,99.052175,99.000000,-0.000050,0.000050,-1
66679,2024-07-01,6682BE1857D0000C0002,Client271,CustomerTimeOut,OFFTHERUN,0,0,0,15,Trader4,...,0.457221,1.000,98.085938,98.095703,98.105469,98.132812,0.000000,0.019531,-0.037109,-1


In [87]:

for trader in df['Trader'].unique():
    print(trader)
    tmp_df = df[df["Trader"] == trader]

    # Step 1: Define the volume bucket size (e.g., 50,000,000)
    V = 50000000

    # Initialize variables
    current_volume = 0
    current_buy_volume = 0
    current_sell_volume = 0
    buckets = []

    # Step 2: Loop through each trade and classify as buy or sell, then aggregate into volume buckets
    for index, row in tmp_df.iterrows():
        trade_volume = row['Market Traded Vol (USD)']
        trade_sign = row['Sign']  # 1 for buy, -1 for sell

        while trade_volume > 0:
            remaining_volume = V - current_volume
            
            if trade_volume >= remaining_volume:
                # Add remaining volume to the current bucket
                if trade_sign == 1:
                    current_buy_volume += remaining_volume
                else:
                    current_sell_volume += remaining_volume
                
                # Add the current bucket to the list
                buckets.append({'Buy Volume': current_buy_volume, 'Sell Volume': current_sell_volume})
                
                # Reset for the next bucket
                trade_volume -= remaining_volume
                current_volume = 0
                current_buy_volume = 0
                current_sell_volume = 0
            else:
                # Add the entire trade volume to the current bucket
                if trade_sign == 1:
                    current_buy_volume += trade_volume
                else:
                    current_sell_volume += trade_volume
                
                current_volume += trade_volume
                trade_volume = 0

    # Step 3: Convert the buckets list into a DataFrame
    buckets_df = pd.DataFrame(buckets)

    # Step 4: Calculate the order imbalance for each bucket
    buckets_df['Order Imbalance'] = abs(buckets_df['Buy Volume'] - buckets_df['Sell Volume'])

    # Step 5: Calculate VPIN using the order imbalances over the number of buckets
    VPIN = buckets_df['Order Imbalance'].sum() / (len(buckets_df) * V)

    # Output VPIN
    print(f"VPIN: {VPIN}")

Trader1
VPIN: 0.6897568640915593
Trader2
VPIN: 0.34017261648745517
Trader3
VPIN: 0.6329120815179199
Trader4
VPIN: 0.863845641819942
Trader5
VPIN: 0.6963518906605922
Trader6
VPIN: 0.6901788753861998
Trader7
VPIN: 0.8305956834532374
Trader8
VPIN: 0.6033470666666667
Trader9
VPIN: 0.8969449325153375
